# Nemotron 3 Diarization — Unseen Baseline Evaluation

**Project:** From Speech to Investment Intelligence (Shark Tank Egypt)  
**Model:** [`nvidia/Nemotron-3-Diarization`](https://huggingface.co/nvidia/Nemotron-3-Diarization) (arXiv:2611.12246)  
**Unseen Benchmarks (Zero Data Leakage):**
1. **ArzEn `test`** (`Mbahaa08/sharktank-diarization-data`) — 13 Egyptian Arabic + English code-switched interviews (~231 min speech)
2. **MSDWild `test`** (`ggfox00000/dia-MsdWild-test`) — Real-world multi-speaker YouTube panels/interviews (`few` & `many` splits)

> Works on **Kaggle (GPU T4 x2 / P100)** and **Google Colab (GPU T4 / L4 / A100)**.

In [ ]:
# 1. Install dependencies (transformers from GitHub main for native Nemotron-3-Diarization support)
!pip install -q --upgrade git+https://github.com/huggingface/transformers
!pip install -q pyannote.metrics soundfile accelerate huggingface_hub pandas tqdm

In [ ]:
# 2. Authenticate with Hugging Face (needed for your private ArzEn repo + faster downloads)
import os
from getpass import getpass
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")

# Try Kaggle Secrets
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass

# Try Colab Secrets
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass

# Fallback: interactive prompt
if not hf_token:
    hf_token = getpass("Enter your Hugging Face Token (Read access): ")

login(token=hf_token)
print("Hugging Face login successful!")

In [ ]:
# 3. Download Unseen Datasets directly onto Kaggle/Colab disk
import json
from pathlib import Path
from huggingface_hub import snapshot_download, hf_hub_download

WORK_DIR = Path("./diar_eval_work")
ARZEN_DIR = WORK_DIR / "arzen_repo"
MSDWILD_DIR = WORK_DIR / "msdwild_test"
OUT_DIR = WORK_DIR / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 3A. Download ArzEn test manifest + RTTMs first, then ONLY the 13 test WAVs (fast!)
print("[1/2] Downloading ArzEn test split from Mbahaa08/sharktank-diarization-data...")
snapshot_download(
    repo_id="Mbahaa08/sharktank-diarization-data",
    repo_type="dataset",
    local_dir=str(ARZEN_DIR),
    allow_patterns=["arzen/test/*", "arzen/test/rttm/*"],
)
arzen_manifest = json.loads((ARZEN_DIR / "arzen/test/manifest.json").read_text(encoding="utf-8"))
test_wav_patterns = [item["audio_path"] for item in arzen_manifest]
snapshot_download(
    repo_id="Mbahaa08/sharktank-diarization-data",
    repo_type="dataset",
    local_dir=str(ARZEN_DIR),
    allow_patterns=test_wav_patterns,
)
print(f"ArzEn test ready: {len(arzen_manifest)} recordings")

# 3B. Download MSDWild test directly from Hugging Face
print("[2/2] Downloading MSDWild test from ggfox00000/dia-MsdWild-test...")
snapshot_download(
    repo_id="ggfox00000/dia-MsdWild-test",
    repo_type="dataset",
    local_dir=str(MSDWILD_DIR),
)
print("MSDWild test ready!")

In [ ]:
# 4. Load nvidia/Nemotron-3-Diarization on GPU
import torch
import soundfile as sf
import numpy as np
from transformers import AutoModelForAudioFrameClassification, AutoProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if (device == "cuda" and torch.cuda.is_bf16_supported()) else (torch.float16 if device == "cuda" else torch.float32)
print(f"Using device: {device} | dtype: {dtype}")
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

MODEL_ID = "nvidia/Nemotron-3-Diarization"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForAudioFrameClassification.from_pretrained(MODEL_ID, torch_dtype=dtype).to(device)
model.eval()
TARGET_SR = processor.feature_extractor.sampling_rate
print(f"Loaded {MODEL_ID} | sampling_rate={TARGET_SR}")

In [ ]:
# 5. Diarization Inference Helper (supports long audio via AOSC chunked inference + short audio offline)
from scipy.signal import resample_poly
from math import gcd

def load_wav_16k(path: Path, target_sr: int = 16000) -> np.ndarray:
    audio, sr = sf.read(str(path), dtype="float32")
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != target_sr:
        g = gcd(sr, target_sr)
        audio = resample_poly(audio, target_sr // g, sr // g).astype(np.float32)
    return audio


def diarize_audio(audio: np.ndarray, offline_max_sec: float = 180.0, streaming_mode: str = "low_latency"):
    """Run Nemotron-3-Diarization and return segments: [{'Start', 'End', 'Speaker'}, ...].

    Uses single-pass offline inference for clips <= offline_max_sec (highest accuracy),
    and chunked AOSC streaming inference for longer recordings (prevents OOM on 16GB T4).
    """
    dur_sec = len(audio) / TARGET_SR

    if dur_sec <= offline_max_sec:
        try:
            inputs = processor(audio, sampling_rate=TARGET_SR).to(model.device, dtype=model.dtype)
            with torch.inference_mode():
                logits = model(**inputs).logits
            segments = processor.extract_speaker_dict(logits, inputs.attention_mask)[0]
            return segments, "offline"
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()

    # Chunked AOSC inference (handles unlimited duration with low VRAM)
    processor.set_streaming_mode(streaming_mode)

    def inputs_generator():
        yield processor(
            audio[: processor.num_samples_first_audio_chunk],
            sampling_rate=TARGET_SR,
            is_streaming=True,
            is_first_audio_chunk=True,
        )
        mel_frame_idx = processor.num_mel_frames_per_step
        start_idx = processor.audio_chunk_start(mel_frame_idx)
        while (end_idx := start_idx + processor.num_samples_per_audio_chunk) <= audio.shape[0]:
            yield processor(
                audio[start_idx:end_idx],
                sampling_rate=TARGET_SR,
                is_streaming=True,
                is_first_audio_chunk=False,
            )
            mel_frame_idx += processor.num_mel_frames_per_step
            start_idx = processor.audio_chunk_start(mel_frame_idx)

        if start_idx < audio.shape[0]:
            yield processor(
                audio[start_idx:],
                sampling_rate=TARGET_SR,
                is_streaming=True,
                is_first_audio_chunk=False,
                is_last_audio_chunk=True,
            )

    speaker_cache, all_logits = None, []
    with torch.inference_mode():
        for inputs in inputs_generator():
            inputs = inputs.to(model.device, dtype=model.dtype)
            outputs = model(**inputs, speaker_cache=speaker_cache)
            all_logits.append(outputs.logits.cpu())
            speaker_cache = outputs.speaker_cache

    full_logits = torch.cat(all_logits, dim=1)
    segments = processor.extract_speaker_dict(full_logits)[0]
    return segments, f"chunked_{streaming_mode}"

In [ ]:
# 6. Standard Academic Metrics (DER, Missed, False Alarm, Confusion, JER) via pyannote.metrics
import time
import pandas as pd
from tqdm.auto import tqdm
from pyannote.core import Annotation, Segment
from pyannote.metrics.diarization import DiarizationErrorRate, JaccardErrorRate


def load_rttm(rttm_path: Path) -> Annotation:
    ann = Annotation(uri=rttm_path.stem)
    with rttm_path.open("r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 8 and parts[0] == "SPEAKER":
                start = float(parts[3])
                dur = float(parts[4])
                spk = parts[7]
                if dur > 0:
                    ann[Segment(start, start + dur)] = spk
    return ann


def segments_to_annotation(segments: list[dict], uri: str) -> Annotation:
    ann = Annotation(uri=uri)
    for seg in segments:
        s = float(seg["Start"])
        e = float(seg["End"])
        if e > s:
            ann[Segment(s, e)] = f"speaker_{seg['Speaker']}"
    return ann


def save_rttm(segments: list[dict], uri: str, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", encoding="utf-8") as f:
        for seg in segments:
            s = float(seg["Start"])
            dur = float(seg["End"]) - s
            if dur > 0:
                f.write(f"SPEAKER {uri} 1 {s:.3f} {dur:.3f} <NA> <NA> speaker_{seg['Speaker']} <NA> <NA>\n")


def evaluate_benchmark(items: list[dict], name: str, max_files: int | None = None):
    """Evaluate Nemotron-3-Diarization on a list of {'id', 'audio_path', 'rttm_path'} items."""
    if max_files is not None:
        items = items[:max_files]

    der_strict = DiarizationErrorRate(collar=0.0, skip_overlap=False)
    der_c25 = DiarizationErrorRate(collar=0.25, skip_overlap=False)
    der_no_ovl = DiarizationErrorRate(collar=0.25, skip_overlap=True)
    jer_metric = JaccardErrorRate(collar=0.0, skip_overlap=False)

    pred_dir = OUT_DIR / f"pred_rttm_{name}"
    rows = []
    total_audio_sec = 0.0
    total_infer_sec = 0.0

    for item in tqdm(items, desc=f"Evaluating {name}"):
        rid = item["id"]
        audio = load_wav_16k(Path(item["audio_path"]), TARGET_SR)
        dur_sec = len(audio) / TARGET_SR
        ref = load_rttm(Path(item["rttm_path"]))

        t0 = time.perf_counter()
        segments, mode = diarize_audio(audio)
        dt = time.perf_counter() - t0
        total_audio_sec += dur_sec
        total_infer_sec += dt

        hyp = segments_to_annotation(segments, uri=rid)
        save_rttm(segments, rid, pred_dir / f"{rid}.rttm")

        comp0 = der_strict(ref, hyp, detailed=True)
        comp25 = der_c25(ref, hyp, detailed=True)
        _ = der_no_ovl(ref, hyp)
        j_val = jer_metric(ref, hyp)

        rows.append({
            "id": rid,
            "mode": mode,
            "dur_sec": round(dur_sec, 1),
            "infer_sec": round(dt, 2),
            "ref_spks": len(ref.labels()),
            "hyp_spks": len(hyp.labels()),
            "DER_0.0s_%": round(comp0["diarization error rate"] * 100, 2),
            "Miss_0.0s_%": round((comp0["missed detection"] / max(comp0["total"], 1e-9)) * 100, 2),
            "FA_0.0s_%": round((comp0["false alarm"] / max(comp0["total"], 1e-9)) * 100, 2),
            "Conf_0.0s_%": round((comp0["confusion"] / max(comp0["total"], 1e-9)) * 100, 2),
            "DER_0.25s_%": round(comp25["diarization error rate"] * 100, 2),
            "JER_0.0s_%": round(j_val * 100, 2),
        })

    df = pd.DataFrame(rows)
    df.to_csv(OUT_DIR / f"{name}_per_file.csv", index=False)

    # Pooled global metrics (standard academic reporting)
    r0 = der_strict.report(display=False).loc["TOTAL"]
    r25 = der_c25.report(display=False).loc["TOTAL"]
    summary = {
        "benchmark": name,
        "num_files": len(items),
        "total_audio_min": round(total_audio_sec / 60.0, 2),
        "RTFx": round(total_audio_sec / max(total_infer_sec, 1e-6), 1),
        "DER_0.0s_with_overlap_%": round(float(abs(der_strict)) * 100, 2),
        "Miss_0.0s_%": round(float(r0[("missed detection", "%")]), 2),
        "FA_0.0s_%": round(float(r0[("false alarm", "%")]), 2),
        "Confusion_0.0s_%": round(float(r0[("confusion", "%")]), 2),
        "DER_0.25s_with_overlap_%": round(float(abs(der_c25)) * 100, 2),
        "DER_0.25s_no_overlap_%": round(float(abs(der_no_ovl)) * 100, 2),
        "JER_0.0s_%": round(float(abs(jer_metric)) * 100, 2),
    }
    return summary, df

In [ ]:
# 7. BASELINE 1: ArzEn test (Egyptian Arabic + English Code-Switched, 13 interviews, 100% Unseen!)
arzen_items = [
    {
        "id": x["recording_id"],
        "audio_path": str(ARZEN_DIR / x["audio_path"]),
        "rttm_path": str(ARZEN_DIR / x["rttm_path"]),
    }
    for x in arzen_manifest
]

arzen_summary, arzen_df = evaluate_benchmark(arzen_items, name="arzen_test")
print("=== ArzEn Test (Egyptian Arabic-English) Baseline ===")
print(json.dumps(arzen_summary, indent=2))
display(arzen_df.sort_values("DER_0.0s_%", ascending=False))

In [ ]:
# 8. BASELINE 2: MSDWild test (Real-world multi-speaker YouTube panels/interviews, 100% Unseen!)
def build_msdwild_items(split_sub: str) -> list[dict]:
    audio_dir = MSDWILD_DIR / "audio" / split_sub
    rttm_dir = MSDWILD_DIR / "rttm"
    items = []
    for wav in sorted(audio_dir.glob("*.wav")):
        # Match rttm either directly in rttm/ or rttm/<split_sub>/
        cand1 = rttm_dir / split_sub / f"{wav.stem}.rttm"
        cand2 = rttm_dir / f"{wav.stem}.rttm"
        rttm = cand1 if cand1.exists() else cand2
        if rttm.exists():
            items.append({"id": wav.stem, "audio_path": str(wav), "rttm_path": str(rttm)})
    return items

msd_few_items = build_msdwild_items("few")
msd_many_items = build_msdwild_items("many")
print(f"MSDWild test found -> few: {len(msd_few_items)} files | many: {len(msd_many_items)} files")

msd_few_summary, msd_few_df = evaluate_benchmark(msd_few_items, name="msdwild_few")
msd_many_summary, msd_many_df = evaluate_benchmark(msd_many_items, name="msdwild_many")

print("=== MSDWild Few Baseline ===")
print(json.dumps(msd_few_summary, indent=2))
print("=== MSDWild Many Baseline ===")
print(json.dumps(msd_many_summary, indent=2))

In [ ]:
# 9. Save Combined Baseline Report
all_summaries = [arzen_summary, msd_few_summary, msd_many_summary]
(OUT_DIR / "baseline_summary.json").write_text(json.dumps(all_summaries, indent=2), encoding="utf-8")
display(pd.DataFrame(all_summaries))